In [1]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    参数:
        city: 城市名称，如"北京"、"上海"

    返回:
        天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"

In [ ]:
##直接调用
get_weather.invoke({'city':'上海'})

'上海晴天，温度 15°C'

In [4]:
##基于模型调用
import base64
from langchain_deepseek import ChatDeepSeek
from langchain.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
class LLM:
    def __init__(self):
        load_dotenv(".env.local",override=True)
        self.api_key= os.getenv("DEEPSEEK_API_KEY").strip()
        self.model= os.getenv("MODEL").strip()
        self.url= os.getenv("DEEPSEEK_BASE_URL").strip()
        try:
            self.thinking_timeout= float(os.getenv("TINKING_TIMEOUT",30).strip())
            self.num_retries= int(os.getenv("RETRY_NUM",2).strip())
            self.temperature= int(os.getenv("TEMPERATURE",1).strip())
            self.stream= True if os.getenv("STREAM").strip() == 'True' else False
            self.max_token= int(os.getenv("MAX_TOKEN").strip()) if os.getenv("MAX_TOKEN").strip() else None
        except Exception as e:
            raise Exception(f'配置文件传入非法参数。错误信息：{e}')
        self.set_llm()
    def set_llm(self):
        self.llm = ChatDeepSeek(
            model= self.model,
            api_key= self.api_key,
            streaming= self.stream,
            api_base= self.url,
            temperature=self.temperature,
            request_timeout= self.thinking_timeout,
            max_tokens= self.max_token,
            max_retries= self.num_retries,
            extra_body={"thinking":{"type":"enabled"}}
            # model_kwargs=   {'tools':[]}##用来存放一些langchain没有列出但模型本身支持的，比如tools
            # configurable_fields= ('model','temperature') ## 用来允许 config中的configurable 覆盖
        )

model = LLM().llm

In [8]:

model_with_tool = model.bind_tools([get_weather])
messages = []
human_input = "上海天气如何"
messages.append({'role':'user','content':human_input})
res = model_with_tool.invoke(messages)
messages.append(res)
for call in res.tool_calls:
    if call['name'] == 'get_weather':
        t_res = get_weather.invoke(call)
        messages.append(t_res)
res = model_with_tool.invoke(messages)
print(res)

content='上海今天天气不错，是 **晴天** ☀️，温度大约 **15°C**，体感会比较舒适。如果想出门的话，是个不错的好天气！不过早晚温差可能较大，建议适当添衣保暖。' additional_kwargs={'reasoning_content': '好的，上海天气信息已经获取到了。让我给用户一个清晰的回答。'} response_metadata={'finish_reason': 'stop', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'model_provider': 'deepseek'} id='lc_run--019fb193-3e41-74e3-8c71-4514da8e16e4' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 391, 'output_tokens': 64, 'total_tokens': 455, 'input_token_details': {'cache_read': 256}, 'output_token_details': {'reasoning': 16}}


# 工具定义

#### 如果在描述中声明了描述，必须带类型

In [17]:
def count(i:int=0):
    """
    工具用于数数
    
    Args:
        i : 具体要数的数
    
    Returns:
        返回要数的数
    """
    return i

# c_model = model.bind_tools([count])
# res = c_model.invoke('数一下3')
# if res.tool_calls:
#     print(res)

### 底层会调 convert_to_openai_tool

In [18]:
from langchain_core.utils.function_calling import convert_to_openai_tool
convert_to_openai_tool(count)

{'type': 'function',
 'function': {'name': 'count',
  'description': '工具用于数数',
  'parameters': {'properties': {'i': {'default': 0,
     'description': '具体要数的数',
     'type': 'integer'}},
   'type': 'object'}}}

In [ ]:
@tool(description='工具用于1数数',parse_docstring=True)
def count(i:int=0):
    """
    工具用于数数
    
    Args:
        i : 具体要数的数
    
    Returns:
        返回要数的数
    """
    return i+1
convert_to_openai_tool(count)

{'type': 'function',
 'function': {'name': 'count',
  'description': '工具用于1数数',
  'parameters': {'properties': {'i': {'default': 0, 'type': 'integer'}},
   'type': 'object'}}}

重命名----不推荐

In [22]:
@tool(name_or_callable='数数',parse_docstring=True)
def count(i:int=0):
    """
    工具用于数数
    
    Args:
        i : 具体要数的数
    
    Returns:
        返回要数的数
    """
    return i+1
convert_to_openai_tool(count)

{'type': 'function',
 'function': {'name': '数数',
  'description': '工具用于数数',
  'parameters': {'properties': {'i': {'default': 0,
     'description': '具体要数的数',
     'type': 'integer'}},
   'type': 'object'}}}

定义args_schema

In [27]:
from pydantic import BaseModel,Field
from typing import Literal
class count_input(BaseModel):
    i : int = Field(
        description= '要数的数字',
        default= 0
    )
    j : Literal[1,2,3] = Field(
        default= 2
    )

@tool(description="数数的工具",args_schema=count_input)
def count(i,j):
    return i+j
convert_to_openai_tool(count)

{'type': 'function',
 'function': {'name': 'count',
  'description': '数数的工具',
  'parameters': {'properties': {'i': {'default': 0,
     'description': '要数的数字',
     'type': 'integer'},
    'j': {'default': 2, 'enum': [1, 2, 3], 'type': 'integer'}},
   'type': 'object'}}}

In [28]:
json_schema = {'properties': {'i': {'default': 0,
     'description': '要数的数字',
     'type': 'integer'},
    'j': {'default': 2, 'enum': [1, 2, 3], 'type': 'integer'}},
   'type': 'object'}
@tool(description="数数的工具",args_schema=json_schema)
def count(i,j):
    return i+j
convert_to_openai_tool(count)

{'type': 'function',
 'function': {'name': 'count',
  'description': '数数的工具',
  'parameters': {'properties': {'i': {'default': 0,
     'description': '要数的数字',
     'type': 'integer'},
    'j': {'default': 2, 'enum': [1, 2, 3], 'type': 'integer'}},
   'type': 'object'}}}

tool_choice()
+ none 禁用
+ auto
+ requied 必用
+ 指定tool名，强制使用tool名指定的tool


In [29]:
b_m = model.bind_tools([get_weather],tool_choice='none')
b_m.invoke('今天北京天气如何')

AIMessage(content='这是一个时效性问题，我无法直接提供今天的实时天气数据。为了获取最准确的北京今日天气（包括温度、风力、降水概率等），建议你**打开联网搜索功能**进行查询，或者直接查看手机自带的天气应用、访问中国天气网等权威气象平台。', additional_kwargs={'reasoning_content': '嗯，用户问的是今天北京的天气情况，这是一个需要实时数据的问题。我的知识截止于2025年5月，没有联网获取最新天气信息的能力。直接给出一个过期或猜测的答案可能会误导用户。最好的做法是诚实地说明我的限制，同时提供可行的解决方案。我可以建议用户开启联网搜索功能，或者推荐一些可靠的气象信息渠道，比如中国天气网、官方气象App或主流天气App。这样既回应了用户的需求，又避免了提供不准确的信息。'}, response_metadata={'finish_reason': 'stop', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'model_provider': 'deepseek'}, id='lc_run--019fb207-27da-7a70-82f5-ee080f3090e6', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 164, 'total_tokens': 172, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 104}})